# Stage 3 solver budget amendment (CPU, once)
Use the same pinned runtime as the original plan. This creates a new plan and a small immutable compatibility receipt; no fits run. Keep other writers idle until this initializer flushes Drive. Repeating this notebook reuses its exact receipt.


In [ ]:
import os, sys, subprocess, json
from pathlib import Path
if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError('Use Python 3.11/3.12 and the frozen CPU requirements.')
for name in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'
from google.colab import drive
drive.mount('/content/drive')
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
REPO = Path('/content/Research-stage3-amendment')
CODE_REF = 'phase1'  # Resolve once below; replace with a published worker SHA to pin explicitly.
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Drazr/Research.git', str(REPO)], check=True)
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean checkout; preserve local edits before continuing.')
subprocess.run(['git', 'fetch', 'origin', CODE_REF], cwd=REPO, check=True)
CODE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'FETCH_HEAD'], cwd=REPO, text=True).strip()
subprocess.run(['git', 'checkout', '--detach', CODE_COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO/'requirements-stage2-cpu.txt')], check=True)
print('Fixed worker commit:', CODE_COMMIT)
print('If installation requests a restart, restart and rerun setup.')


In [ ]:
LEGACY_PLAN = PHASE1_ROOT / 'cache/shared/stage3-plans/20261116T092755Z-6c60898a26284408b8e5c717a5653e0a/_COMPLETE.json'
DIAGNOSTIC = PHASE1_ROOT / 'results/stage3-solver-diagnostics/20261117T010451Z-ce2afa1e3d084eabb8ef22a02e9d9396/_COMPLETE.json'
LEGACY_HANDOFF = PHASE1_ROOT / 'configs/stage3-profile-handoff-6749bd9f53148def.json'
HANDOFF = PHASE1_ROOT / 'configs/stage3-budget-amendment-6749bd9f53148def.json'


In [ ]:
for value in (LEGACY_PLAN, DIAGNOSTIC, LEGACY_HANDOFF):
    if not value.is_file(): raise RuntimeError('Missing mounted input: '+str(value))
if subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()!=CODE_COMMIT or subprocess.check_output(['git','status','--porcelain'],cwd=REPO,text=True).strip():
    raise RuntimeError('Checkout changed after setup.')
child = "import hashlib,json,sys\nfrom pathlib import Path\nfrom src.stage3.interfaces import amend_solver_budget,load_fit_plan,current_producer\nfrom src.stage3.artifacts import Store\nfrom src.stage3.recovery import load_run_index\nfrom src.stage2.schema import plain,check\nroot,legacy,diagnostic,oldhandoff,output=map(Path,sys.argv[1:])\nprotocol=json.loads(Path('configs/protocol.json').read_text())\nproducer=current_producer('persistent-drive')\nstore=Store(root,protocol)\nraw=oldhandoff.read_bytes(); h=json.loads(raw)\ncheck(root/h['plan']==legacy,'Wrong original handoff plan')\nlegacy_ref=store.ref(legacy); diagnostic_ref=store.ref(diagnostic)\nif output.exists():\n    receipt=json.loads(output.read_text())\n    check(receipt['legacy_handoff_sha256']==hashlib.sha256(raw).hexdigest()\n          and receipt['legacy_plan']==legacy_ref and receipt['diagnostic']==diagnostic_ref,'Existing handoff differs')\n    plan=load_fit_plan(receipt['plan'],root,protocol)\n    check(plain(plan['compatibility']['legacy_plan'])==legacy_ref\n          and plain(plan['compatibility']['diagnostic'])==diagnostic_ref,'Wrong amended plan')\nelse:\n    plan=amend_solver_budget(legacy,diagnostic,root,protocol,producer)\n    receipt={'schema':'stage3.budget-amendment-handoff.v1','plan':plain(plan.reference),\n             'legacy_plan':legacy_ref,'diagnostic':diagnostic_ref,\n             'legacy_handoff_sha256':hashlib.sha256(raw).hexdigest(),\n             'assignments':h['assignments'],'ownership':h['ownership']}\n    with output.open('x') as f:json.dump(receipt,f,indent=2)\n# One serial initializer prevents duplicate shared directory creation by Drive writers.\njob=plan['fit_identity'][:16]\nfolders=['cache/'+job+'/preprocessing','checkpoints/'+job+'/selections',\n         'predictions/'+job+'/g0','results/'+job+'/g0','results/'+job+'/indexes']\nfolders+=['checkpoints/'+job+'/'+gid[:16] for gid in plan['groups']]\nfor folder in folders:(root/folder).mkdir(parents=True,exist_ok=True)\ncheck(json.loads(output.read_text())==receipt,'Handoff readback differs')\nprint(json.dumps({'status':'passed','plan':str(root/plan.reference['path']),\n                 'handoff':str(output),'fit_identity':plan['fit_identity'],\n                 'stage3_complete':False}),flush=True)\n"
subprocess.run([sys.executable,'-u','-c',child,str(PHASE1_ROOT),str(LEGACY_PLAN),str(DIAGNOSTIC),str(LEGACY_HANDOFF),str(HANDOFF)],cwd=REPO,check=True)
from google.colab import drive
drive.flush_and_unmount()
print('Initialization finished. Remount Drive in worker runtimes before resuming.')
